# Harmonization: BoT-IoT → ตารางกลาง

สมุดนี้มี mapping และโค้ดสร้างตารางอยู่ใน notebook โดยตรง ตามรูปแบบของ NSL-KDD/UNSW-NB15 ไม่เรียก `run_harmonization()` จากไฟล์ `.py`

อ่านไฟล์ cleaned แยก train/test แล้วสร้าง 3 ตารางที่เชื่อมด้วย `record_id`:

- `features.csv`: คุณลักษณะการเชื่อมต่อ; ไม่มี label
- `labels.csv`: ป้ายกำกับจริงและประเภทการโจมตี
- `metadata.csv`: ต้นทาง split และ flags สำหรับตามรอย

ฟิลด์ที่ dataset นี้ไม่มีจะเก็บเป็น missing พร้อม availability flag ไม่สร้างค่าปลอมขึ้นมา


## 1. กำหนด path และตรวจผล cleansing

หาโฟลเดอร์จากตำแหน่ง notebook และหยุดหากไม่มีไฟล์ cleaned หรือ manifest ที่ขั้นก่อนหน้าควรสร้าง.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

DATASET_NAME = 'BoT-IoT'
RAW_FILES = {"train": 'BoT_IoT_training-set.csv', "test": 'BoT_IoT_testing-set.csv'}

def find_dataset_dir(start: Path, dataset_name: str) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == dataset_name and (candidate / "download.py").is_file():
            return candidate
        nested = candidate / "data-pipeline" / "sub-dataset" / dataset_name
        if (nested / "download.py").is_file():
            return nested
    raise FileNotFoundError(f"หาโฟลเดอร์ {dataset_name} จาก {start} ไม่พบ")

DATASET_DIR = find_dataset_dir(Path.cwd(), DATASET_NAME)
RAW_DIR = DATASET_DIR / "raw"
OUTPUT_DIR = DATASET_DIR / "output"
CLEAN_DIR = OUTPUT_DIR / "cleansing"
HARM_DIR = OUTPUT_DIR / "harmonized"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Dataset:", DATASET_NAME)
print("Cleansing input:", CLEAN_DIR)
print("Harmonized output:", HARM_DIR)

clean_manifest_path = CLEAN_DIR / "source_manifest.csv"
required_clean = [CLEAN_DIR / filename for filename in RAW_FILES.values()]
missing_clean = [path.name for path in required_clean if not path.is_file()]
if missing_clean or not clean_manifest_path.is_file():
    raise FileNotFoundError(f"ผล cleansing ไม่ครบ: {missing_clean}; รัน cleansing.ipynb ก่อน")
HARM_DIR.mkdir(parents=True, exist_ok=True)


## 2. กำหนดกติกา mapping

แปลง protocol, port/service และ connection state ให้เป็นกลุ่มมาตรฐานที่ใช้ร่วมกับ dataset อื่น.

In [ ]:
def map_port_to_service(port: int) -> str:
    port = int(port) if pd.notna(port) else -1
    if port == 53:
        return "dns"
    if port in (20, 21):
        return "ftp"
    if port in (80, 8080):
        return "http"
    if port == 443 or (DATASET_NAME == "BoT-IoT" and port == 8883):
        return "https_ssl"
    if port == 25:
        return "smtp"
    if port == 22 or (DATASET_NAME == "BoT-IoT" and port == 23):
        return "ssh"
    return "other"

def map_state_to_group(state_value: str) -> str:
    state = str(state_value).upper()
    if state in ("CON", "FIN", "ESTABLISHED"):
        return "established"
    if state in ("REQ", "INT", "NO-REPLY"):
        return "no-reply"
    if state in ("RST", "REJ", "RESET"):
        return "reset"
    return "other"

def normalize_category(values: pd.Series, allowed: list[str]) -> pd.Series:
    lowered = values.astype(str).str.lower()
    return lowered.where(lowered.isin(allowed), "other")


## 3. แปลงข้อมูลทีละ split เป็น features / labels / metadata

ตัวอย่าง mapping: `sbytes` หรือ `total_fwd_bytes` → `bytes_forward`; label แยกไว้ในตาราง labels.

In [ ]:
feature_parts, label_parts, metadata_parts = [], [], []

for split_name, filename in RAW_FILES.items():
    frame = pd.read_csv(CLEAN_DIR / filename, low_memory=False)
    record_ids = [f"{DATASET_NAME}:{split_name}:{row_id}" for row_id in frame["orig_row_id"]]

    # BoT-IoT field names -> shared flow fields
    service = frame["dport"].apply(map_port_to_service)
    connection_state = frame["state"].apply(map_state_to_group)
    protocol = normalize_category(frame["proto"], ["tcp", "udp", "icmp"])
    duration = frame["dur"].fillna(0).astype(float)
    bytes_forward = frame["sbytes"].fillna(0).astype(float)
    bytes_backward = frame["dbytes"].fillna(0).astype(float)
    packets_forward = frame["spkts"].fillna(0).astype(float)
    packets_backward = frame["dpkts"].fillna(0).astype(float)
    packet_total = packets_forward + packets_backward
    mean_packet_size = np.where(packet_total > 0, (bytes_forward + bytes_backward) / packet_total, np.nan)

    feature_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "duration": duration,
        "protocol": protocol,
        "bytes_forward": bytes_forward,
        "bytes_backward": bytes_backward,
        "service_group": service,
        "conn_state_group": connection_state,
        "packets_forward": packets_forward,
        "packets_backward": packets_backward,
        "mean_packet_size": mean_packet_size,
        "ttl_forward": np.nan,
        "ttl_backward": np.nan,
        "tcp_window_forward": np.nan,
        "tcp_window_backward": np.nan,
        "nsl_hot": np.nan,
        "nsl_logged_in": np.nan,
        "nsl_failed_logins": np.nan,
        "nsl_root_shell": np.nan,
        "nsl_count_2s": np.nan,
        "nsl_srv_count_2s": np.nan,
        "nsl_same_srv_rate_2s": np.nan,
        "nsl_serror_rate_2s": np.nan,
    }))

    source_category = frame["category"].astype(str)
    label_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "label_raw": frame["subcategory"].astype(str),
        "label_source_category": source_category,
        "attack_category": source_category,
        "label_binary": frame["attack"].astype(int),
    }))

    metadata_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "source_dataset": DATASET_NAME,
        "orig_row_id": frame["orig_row_id"],
        "source_split": split_name,
        "capture_time": pd.NA,
        "split_group": split_name,
        "dedup_flag": frame["_duplicate_flag"],
        "sample_weight": 1.0,
        "has_duration": True,
        "invalid_duration": False,
        "has_protocol": True,
        "has_directional_bytes": True,
        "invalid_bytes_forward": False,
        "invalid_bytes_backward": False,
        "has_capture_time": False,
        "has_service": True,
        "has_connection_state": True,
        "has_packet_stats": True,
        "has_tcp_details": False,
        "has_nsl_content": False,
        "has_nsl_window": False,
        "is_kddtest21_view": False,
    }))


## 4. ตรวจ ID และความสอดคล้องก่อนบันทึก

ยืนยันว่า ID ไม่ซ้ำและเรียงตรงกันทั้งสามตาราง ก่อน export.

In [ ]:
features = pd.concat(feature_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)

if not (features["record_id"].is_unique and labels["record_id"].is_unique and metadata["record_id"].is_unique):
    raise ValueError("record_id ซ้ำในตาราง harmonized")
if not (features["record_id"].tolist() == labels["record_id"].tolist() == metadata["record_id"].tolist()):
    raise ValueError("record_id เรียงไม่ตรงกันใน features, labels และ metadata")

print("Feature table:", features.shape)
print("Label table:", labels.shape)
print("Metadata table:", metadata.shape)
display(labels["attack_category"].value_counts(dropna=False).rename("rows").to_frame())


## 5. บันทึกผลและ provenance

In [ ]:
tables = {"features": features, "labels": labels, "metadata": metadata}
for table_name, table in tables.items():
    table.to_csv(HARM_DIR / f"{table_name}.csv", index=False)

report = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in tables.items()
])
report.to_csv(HARM_DIR / "harmonization_report.csv", index=False)

quality = pd.DataFrame([
    {"source_dataset": DATASET_NAME, "split": split, "bad_duration": 0, "bad_forward": 0, "bad_backward": 0}
    for split in RAW_FILES
])
quality.to_csv(HARM_DIR / "harmonization_quality.csv", index=False)
pd.read_csv(clean_manifest_path).to_csv(HARM_DIR / "source_manifest.csv", index=False)

provenance = {
    "dataset": DATASET_NAME,
    "pipeline_version": "quad_v1_notebook",
    "features_rows": len(features),
    "labels_rows": len(labels),
    "metadata_rows": len(metadata),
    "tables_verified": True,
}
(HARM_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Saved files:")
for path in sorted(HARM_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")
